In [23]:
import os, sys
sys.path.append("../")
from own_utils.oss_new import OssUtil
from own_utils.sql_python_utils import *
from own_utils.H5_utils import H5_utils
import json
import numpy as np
from multiprocessing import Pool
import pandas as pd

In [24]:
country = 'indonesia'
environment = "production"
table_buried_point = 'id_sdk_liveness_buried_points_202503' 
table_detection = 'id_sdk_liveness_detection'
oss = OssUtil(country = country, environment = environment)
connect = Connector(country = country, environment = environment, database = "cv")
h5 = H5_utils(connect = connect, oss = oss, table_buried_point = table_buried_point, table_detection = table_detection)

This created instance is production bucket for id-cv-data country for indonesia


In [25]:
def mediapipe106_h5_file_analysis(id):
    try:
        h5_file_detail = oss.get_h5_file(oss_id= id).split("\n")
        client = None
        backend = None
        detector_loading = []
        pfld_106_loading = []
        detector_init = []
        pfld_106_init = []
        detector_prediction = []
        pfld_106_prediction = []
        overall_processing = []
        for line in h5_file_detail:
            if 'client_type' in line:
                client = line.split('client_type')[-1].strip()
            if 'Backend' in line:
                backend = line.split('Backend')[-1].strip() 
            if 'detector init' in line:
                detector_init.append(int(line.split('detector init')[-1].strip()))
            if '106_pfld init' in line:
                pfld_106_init.append(int(line.split('106_pfld init')[-1].strip()))
            if '106_pfld loading ' in line:
                pfld_106_loading.append(int(line.split('106_pfld loading')[-1].strip()))
            if 'detector loading' in line:
                detector_loading.append(int(line.split('detector loading')[-1].strip()))
            if 'dectector prediction' in line:
                detector_prediction.append(int(line.split('dectector prediction')[-1].strip()))
            if '106_plfd prediction' in line:
                pfld_106_prediction.append(int(line.split('106_plfd prediction')[-1].strip()))
            if 'overall processing' in line:
                overall_processing.append(int(line.split('overall processing')[-1].strip()))
        
        detector_loading = int(np.mean(detector_loading)) if detector_loading else 'not found'
        pfld_106_loading = int(np.mean(pfld_106_loading)) if pfld_106_loading else 'not found'
        detector_init = int(np.mean(detector_init)) if detector_init else 'not found'
        pfld_106_init = int(np.mean(pfld_106_init)) if pfld_106_init else 'not found'
        detector_prediction = int(np.mean(detector_prediction)) if detector_prediction else 'not found'
        pfld_106_prediction = int(np.mean(pfld_106_prediction)) if pfld_106_prediction else 'not found'
        overall_processing = int(np.mean(overall_processing)) if overall_processing else 'not found'
        
        return (client, backend, detector_loading, pfld_106_loading, detector_init, pfld_106_init, detector_prediction, pfld_106_prediction, overall_processing)
    except Exception:
        return None

In [26]:
def process_row(row_input):
    # Assuming 'row' is a dictionary-like object containing necessary data
    index, row, new_columns = row_input
    result = None    
    if isinstance(row['H5_file'], str) and row['H5_file'].endswith(".txt") and row['H5_file'] != "no H5 file":
        oss_id = row['H5_file']
        result = mediapipe106_h5_file_analysis(id = oss_id)
        if not result:
            result = ['no H5 file'] * len(new_columns) 
    else:
        result = ['no H5 file'] * len(new_columns)
        
    return (index, result)

def multiprocess_row(df, new_columns, number_worker = 4):
    df = df.reset_index(drop= True)
    tasks = [(index, row.to_dict(), new_columns) for index, row in df.iterrows()]

    # Set up multiprocessing
    with Pool(processes = number_worker) as pool:
        results = pool.map(process_row, tasks)

    # Update DataFrame with results
    for index, update in results:
        df.loc[index, new_columns] = update

    return df

In [30]:
query = f"""
SELECT *
FROM cv.id_sdk_liveness_result
WHERE user_id in ("zox6656bc", "zoxbcc287")
"""
liveness_data = connect.query(query=query)

print(f"Shape : {liveness_data.shape}")
liveness_data.drop_duplicates(subset= ['liveness_id'], inplace= True, keep= 'last')
liveness_data['data'] =  liveness_data['data'].apply(lambda x: json.loads(x))
liveness_data['ext_info'] =  liveness_data['ext_info'].apply(lambda x: json.loads(x))
liveness_data['liveness_result_msg'] = liveness_data['ext_info'].apply(lambda x: x.get("errMsg", None))
liveness_data['liveness_result_code'] = liveness_data['ext_info'].apply(lambda x: x.get("code", None))
liveness_data['sdk_version'] = liveness_data['ext_info'].apply(lambda x: x.get("sdk_version", None))
liveness_data['system'] = liveness_data['ext_info'].apply(lambda x: x.get("system", None))
liveness_data['deviceInfo'] = liveness_data['ext_info'].apply(lambda x: x.get("deviceInfo", {}).get('source', None))
liveness_result_df = liveness_data[['liveness_id', "create_time", 'partner_id', 'user_id', 'liveness_result_msg', 'liveness_result_code', 'sdk_version', 'system', 'deviceInfo']]
liveness_count_df = liveness_result_df.groupby('user_id').size().reset_index(name = 'user_liveness_count')
liveness_result_df = liveness_result_df.merge(right= liveness_count_df, on= 'user_id', how= 'inner')
user_id_unique = str(tuple(liveness_result_df['user_id'].unique()))
liveness_id_unique = str(tuple(liveness_result_df['liveness_id'].unique()))
buried_detail_df = h5.get_buried_detail(liveness_id= liveness_id_unique)
df = liveness_result_df.merge(buried_detail_df, on = 'liveness_id', how = 'left')
df = df[df["deviceInfo"] == "android-app"].drop(columns = ["detector_type", "create_time", "user_liveness_count", "partner_id"]).reset_index(drop = True)
df = df.drop_duplicates(subset= ['liveness_id'], keep= 'last').reset_index(drop = True)
new_column = ['client', 'backend', 'detector_loading', "pfld_106_loading", "detector_init", "pfld_106_init", "detector_prediction", "pfld_106_prediction", "overall_processing"]
result = None
for index, row in df.iterrows():
    oss_id = row['H5_file']
    result = mediapipe106_h5_file_analysis(id = oss_id)
    df.loc[index, new_column] = result
df

Shape : (2, 12)
Getting buried detail................
Getting buried detail done!!!


,liveness_id,user_id,liveness_result_msg,liveness_result_code,sdk_version,system,deviceInfo,H5_file,client,backend,detector_loading,pfld_106_loading,detector_init,pfld_106_init,detector_prediction,pfld_106_prediction,overall_processing
0,56fcf445-2b80-4923-8564-96e995686c21,zox6656bc,活体认证成功,200,2.2.0.14,h5,android-app,333911b670ea340f9ead2689a3fbca4b.txt,android-app,wasm,5146.0,1218.0,1158.0,1075.0,190.0,592.0,823.0
1,62acc9dc-d440-491f-81e9-963bc5813c42,zoxbcc287,活体认证成功,200,2.2.0.14,h5,android-app,3380093a73dd31c4a930a042ccffcfa6.txt,android-app,wasm,3484.0,337.0,606.0,871.0,164.0,573.0,773.0


In [31]:
print(oss.get_h5_file(oss_id= "333911b670ea340f9ead2689a3fbca4b.txt"))

进入活体次数：10
client_type android-app
detector_type mediapipe106
system_info mozilla/5.0 (windows nt 10.0; win64; x64; rv:54.0) gecko/20100101 firefox/54.0
鉴权参数 partner-code:cv timestamp:1742872934315 token:698cc01f9532206f6ba26511b151cfbe userId:null device:android-app
modeltype: mediapipe_detector_106
监测到当前网络速度较慢,最多需要,S,请耐心等待
getUserMedia
here1
打开相机
270 camera_width 360 camera_height
medieRecoderFlag true
startCompression
开始录制 
audioinput:  (ID: )
videoinput: camera2 1, facing front (ID: 940cd1f47d150b4aa7b3060378149177cafdbc6eee1d5dcf0ee00ab71a05c160)
videoinput: camera2 0, facing back (ID: c8be0ee324c0b3e447af666bc7ddd6687b63fc28f413fabd4996faa8efa933ca)
audiooutput:  (ID: )
res [object Object],[object Object],[object Object]
color rgb(247,206,75)
color rgb(70,70,70)
color rgb(15,255,15)
17259 length
Backend wasm
106_pfld loading 1218
I0325 03:22:29.192000 1922656 gl_context.cc:386] GL version: 3.0 (OpenGL ES 3.0 (WebGL 2.0 (OpenGL ES 3.0 Chromium))), renderer: WebKit WebGL
W0325 03:22

In [29]:
query = f"""
SELECT *
FROM cv.id_sdk_liveness_buried_points_202503
WHERE user_id = "zox6656bc"
"""
buried_data = connect.query(query=query)
buried_data.loc[0]['data']

'第47次活体。手机品牌：samsung，手机型号：SM-A042F，系统版本：13，系统sdk版本：33，系统语言：zh，CPU架构：arm64-v8a，GPU渲染器：PowerVR Rogue GE8320，GPU供应商：Imagination Technologies，GPU版本：OpenGL ES-CM 1.1，WebView版本：131.0.6778.200'

In [ ]:
# query = f"""
# SELECT *
# FROM cv.id_sdk_liveness_result
# WHERE create_time >= '2025-03-01'
# AND create_time < '2025-03-21'
# AND partner_id in (167)
# """
# liveness_data = connect.query(query=query)

# print(f"Shape : {liveness_data.shape}")
# liveness_data.drop_duplicates(subset= ['liveness_id'], inplace= True, keep= 'last')
# liveness_data['data'] =  liveness_data['data'].apply(lambda x: json.loads(x))
# liveness_data['ext_info'] =  liveness_data['ext_info'].apply(lambda x: json.loads(x))
# liveness_data['liveness_result_msg'] = liveness_data['ext_info'].apply(lambda x: x.get("errMsg", None))
# liveness_data['liveness_result_code'] = liveness_data['ext_info'].apply(lambda x: x.get("code", None))
# liveness_data['sdk_version'] = liveness_data['ext_info'].apply(lambda x: x.get("sdk_version", None))
# liveness_data['system'] = liveness_data['ext_info'].apply(lambda x: x.get("system", None))
# liveness_data['deviceInfo'] = liveness_data['ext_info'].apply(lambda x: x.get("deviceInfo", {}).get('source', None))
# liveness_result_df = liveness_data[['liveness_id', "create_time", 'partner_id', 'user_id', 'liveness_result_msg', 'liveness_result_code', 'sdk_version', 'system', 'deviceInfo']]
# liveness_count_df = liveness_result_df.groupby('user_id').size().reset_index(name = 'user_liveness_count')
# liveness_result_df = liveness_result_df.merge(right= liveness_count_df, on= 'user_id', how= 'inner')
# user_id_unique = str(tuple(liveness_result_df['user_id'].unique()))
# liveness_id_unique = str(tuple(liveness_result_df['liveness_id'].unique()))
# buried_detail_df = h5.get_buried_detail(liveness_id= liveness_id_unique)
# df = liveness_result_df.merge(buried_detail_df, on = 'liveness_id', how = 'left')
# df = df[df["deviceInfo"] == "android-app"].drop(columns = ["detector_type", "create_time", "user_liveness_count", "partner_id"]).reset_index(drop = True)
# df = df.drop_duplicates(subset= ['liveness_id'], keep= 'last').reset_index(drop = True)

In [12]:
df = pd.read_csv("./data/benchmark.csv")

In [13]:
df

,liveness_id,user_id,liveness_result_msg,liveness_result_code,sdk_version,system,deviceInfo,H5_file
0,9213c4fd-c829-46e7-9567-923ce4803cd1,99E954E9-9FAD-492B-B90D-A66EBE8A314B,DETECTSUCCESS-waktu habis,40003,2.2.0.14,h5,android-app,00272471392739e88ac13604d103b919.txt
1,68ad951c-ed33-43c9-8020-64ef68c6a24a,1169736,ACTIONDOWN-waktu habis,40003,2.2.0.14,h5,android-app,8c5ed90319a5373b82c49860e80da77c.txt
2,fbb452bc-b157-4b83-94c8-42089a1a3c4c,1169989,Deteksi liveness sukses,200,2.2.0.14,h5,android-app,5a56e20ca2f83350ad17d0c6ff7a6c69.txt
3,8b801033-5d18-4f0c-83a7-efe2a7ea9d5c,1170047,Deteksi liveness sukses,200,2.2.0.14,h5,android-app,7176dab2763b34f0941af38363ccd63c.txt
4,723744ef-b9c1-4490-834d-3ec5e47ec3f6,99E954E9-9FAD-492B-B90D-A66EBE8A314B,Deteksi liveness sukses,200,2.2.0.14,h5,android-app,eea35773d7a730f5b3cdcf3e36ab6541.txt
...,...,...,...,...,...,...,...,...
25981,ebb819ca-784a-4fe8-8b7d-9a5bc0e158f9,1373173,Deteksi liveness sukses,200,2.2.0.14,h5,android-app,c09c6d37eb3333e8a5209e75abce88b2.txt
25982,e37304ca-61fa-43f6-a1f4-2c03aeac589c,1346069,Deteksi liveness sukses,200,2.2.0.14,h5,android-app,cf2f67f4d87f3e959fc82706e8d9d9ef.txt
25983,954f8c11-f057-488e-82f2-5d35f6e9eda4,846367,FACENODEFINE-waktu habis,40003,2.2.0.14,h5,android-app,50d0cf15e1d03741bec2570f1070f1d1.txt
25984,795563e2-7f74-4a86-8f72-0054eb5f6667,1373294,Deteksi liveness sukses,200,2.2.0.14,h5,android-app,c1af16d38bff3d2fa5933623c221c3e1.txt


In [14]:
new_column = ['client', 'backend', 'detector_loading', "pfld_106_loading", "detector_init", "pfld_106_init", "detector_prediction", "pfld_106_prediction", "overall_processing"]
df = multiprocess_row(df = df, new_columns = new_column, number_worker = 8)


/tmp/ipykernel_95659/3402720250.py:25: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'not found' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[index, new_columns] = update
/tmp/ipykernel_95659/3402720250.py:25: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'not found' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[index, new_columns] = update
/tmp/ipykernel_95659/3402720250.py:25: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'not found' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[index, new_columns] = update
/tmp/ipykernel_95659/3402720250.py:25: FutureWarning: Setting an item of incompati

In [19]:
df.to_csv("./data/processed_benchmark.csv", index= False)